Digital Business University of Applied Sciences

Data Science und Management (M. Sc.)

DAMI01 / DATA01 Data Analytics

Prof. Dr. Daniel Ambach

Julia Schmid (200022)

***
# Time Series Clustering
# Teil 2: Data Preparation
***


In [1]:
# Imports
import pandas as pd
import numpy as np
import webbrowser
import os
from ydata_profiling import ProfileReport
from IPython.display import display

# Imports Parameter und Funktionen
from parameter import (
    INPUT_PATH,
    MAP_DESCRIPTION,
    MAP_MERCHANT_STATE,
    MAP_ERROR_CATEGORY
)
from funktionen import get_key_value

/Users/juliaschmid/.pyenv/versions/3.11.0/lib/python3.11/site-packages/tslearn/bases/bases.py:15: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


In [2]:
# Einstellung, sodass alle Spalten eines Datensatzes angezeigt werden
pd.set_option("display.max_columns", None)

In [3]:
# Daten laden
path_temp = INPUT_PATH / "data_acquisition.csv"
df = pd.read_csv(path_temp)

***
## Überblick


In [4]:
# Anzahl der Zeilen und Spalten ausgeben
print(f"Anzahl Zeilen: {df.shape[0]}")
print(f"Anzahl Spalten: {df.shape[1]}")

Anzahl Zeilen: 51115337
Anzahl Spalten: 39


In [5]:
# Zehn zufällige Einträge ausgeben
df.sample(n=10)

,id_transaction,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,description,id_cards,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,id_user,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
2523553,8250486,2010-07-13 08:42:00,34,3342,$68.00,Swipe Transaction,43293,Sacramento,CA,95829.0,5499,NaN,Miscellaneous Food Stores,3342,Mastercard,Debit (Prepaid),5672012369002302,02/2020,598,YES,2,$105,06/2007,2013,No,34,41,55,1978,8,Male,7467 Spruce Drive,38.48,-121.34,$25431,$51854,$72162,737,2
29975297,16955745,2015-11-16 00:42:00,1301,3136,$205.95,Chip Transaction,30286,Novato,CA,94947.0,4814,NaN,Telecommunication Services,3189,Mastercard,Debit,5975006425756630,12/2023,551,YES,1,$22297,05/2017,2017,No,1301,26,66,1993,5,Female,9530 Washington Street,38.10,-122.63,$30335,$61850,$81158,728,9
42731625,21056573,2018-04-06 20:07:00,764,3727,$66.34,Chip Transaction,59935,Warner Robins,GA,31088.0,5499,NaN,Miscellaneous Food Stores,286,Visa,Debit,4127768354829504,04/2020,462,YES,1,$16109,01/2011,2011,No,764,77,68,1943,1,Female,6864 Norfolk Boulevard,32.61,-83.63,$20835,$31066,$20902,712,4
8947874,10265541,2011-10-30 15:15:00,20,5535,$156.73,Online Transaction,72813,ONLINE,NaN,NaN,6300,NaN,"Insurance Sales, Underwriting",376,Visa,Credit,4812182592435267,02/2022,213,YES,1,$13000,01/2018,2018,No,20,86,67,1933,12,Female,3631 Plum Boulevard,32.42,-97.10,$19477,$23371,$0,757,8
30872840,17242617,2016-01-16 03:57:00,1274,3149,$43.12,Swipe Transaction,26909,Pawtucket,RI,2860.0,5211,NaN,Lumber and Building Materials,1075,Visa,Debit,4799397143962163,12/2020,310,YES,2,$17537,02/2005,2010,No,1274,49,61,1970,4,Female,824 West Drive,41.95,-71.70,$24260,$49464,$123198,705,6
25650451,15572208,2015-01-22 17:09:00,110,4756,$230.94,Swipe Transaction,51300,Mumbai,India,NaN,3359,NaN,Non-Ferrous Metal Foundries,5189,Mastercard,Debit,5386335706439778,01/2022,713,YES,2,$13323,10/2010,2010,No,110,52,72,1967,10,Female,31204 El Camino Lane,30.26,-97.74,$25594,$52186,$42852,737,4
16333391,12604177,2013-04-12 22:50:00,1274,1075,$10.17,Swipe Transaction,58739,Pascoag,RI,2859.0,5411,NaN,"Grocery Stores, Supermarkets",2905,Mastercard,Debit,5679478018180576,03/2017,465,YES,2,$5391,05/2006,2013,No,1274,49,61,1970,4,Female,824 West Drive,41.95,-71.70,$24260,$49464,$123198,705,6
10347916,10708247,2012-02-09 03:34:00,734,1252,$15.72,Swipe Transaction,20561,Buffalo,NY,14215.0,5912,NaN,Drug Stores and Pharmacies,4990,Mastercard,Credit,5146631536689214,07/2020,498,YES,2,$6300,10/2003,2010,No,734,43,61,1976,7,Female,57799 Oak Boulevard,42.88,-78.85,$12220,$24917,$36101,693,2
23893245,15010031,2014-09-22 17:27:00,1492,4818,$25.40,Swipe Transaction,75936,Bothell,WA,98011.0,5814,NaN,Fast Food Restaurants,2615,Visa,Debit (Prepaid),4860320285002452,12/2023,366,YES,2,$84,04/2009,2017,No,1492,63,58,1957,2,Female,2883 Ocean View Boulevard,47.32,-121.99,$32697,$56635,$13015,786,8
16512689,12661449,2013-04-25 15:23:00,1135,3081,$26.60,Online Transaction,16798,ONLINE,NaN,NaN,4121,NaN,Taxicabs and Limousines,1419,Amex,Credit,335380246966517,11/2020,42,YES,3,$11700,02/2020,2020,No,1135,89,69,1931,1,Female,796 West Lane,33.84,-118.35,$31496,$68010,$0,704,7


In [6]:
# Datensatz-Info ausgeben
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51115337 entries, 0 to 51115336
Data columns (total 39 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   id_transaction         int64  
 1   date                   object 
 2   client_id              int64  
 3   card_id                int64  
 4   amount                 object 
 5   use_chip               object 
 6   merchant_id            int64  
 7   merchant_city          object 
 8   merchant_state         object 
 9   zip                    float64
 10  mcc                    int64  
 11  errors                 object 
 12  description            object 
 13  id_cards               int64  
 14  card_brand             object 
 15  card_type              object 
 16  card_number            int64  
 17  expires                object 
 18  cvv                    int64  
 19  has_chip               object 
 20  num_cards_issued       int64  
 21  credit_limit           object 
 22  acct_open_date  

***
## Phase 4: Datenvorbereitung
***
### **Datenqualität**

### Duplikate

In [7]:
# Anzahl der Duplikate bestimmen
df_duplicates = df[df.duplicated()]
print(f"Dieser Datensatz besitz {len(df_duplicates)} Duplikate.")

Dieser Datensatz besitz 0 Duplikate.


### Fehlende Werte

In [8]:
# Fehlende Werte ermitteln
count_nan = df.isna().sum()

# Prozentsatz der NaN-Werte berechnen
percent_nan = round((count_nan / len(df)) * 100, 2)

# Werte in einem DataFrame speichern
df_nan = pd.DataFrame({
    "Anzahl NaN": count_nan,
    "Prozent NaN": percent_nan
}).sort_values(by="Anzahl NaN", ascending=False)

# Variablen mit fehlenden Werten inkl. Anzahl und Prozentanteil ausgeben
df_nan[df_nan["Anzahl NaN"] > 0]

,Anzahl NaN,Prozent NaN
errors,50306417,98.42
zip,6281710,12.29
merchant_state,5931904,11.60


Fehlende Werte treten ausschließlich bei kategorischen Variablen auf. Diese werden daher durch die Kategorien *Unknown* oder *None* ersetzt.

Bei der Variable **errors** lassen sich fehlende Werte inhaltlich als „kein Fehler aufgetreten" deuten. Sie werden daher mit dem Wert *None* gefüllt.

Bei den Variablen **zip** und **merchant_state** gibt es hingegen keinen inhaltlichen Anhaltspunkt, welcher konkrete Wert hinter den fehlenden Einträgen stecken könnte. Diese werden daher mit *Unknown* gekennzeichnet.

In [9]:
# Fehlende Werte der Variable "errors" als "None" kennzeichnen
df["errors"] = df["errors"].fillna("None")

# Variable "zip" als String speichern und
# fehlende Werte als "Unknown" kennzeichnen
df["zip"] = (
    pd.to_numeric(df["zip"], errors="coerce")
    .astype("Int64")
    .astype("string"))
df["zip"] = df["zip"].fillna("Unknown")

# Fehlende Werte der Variable "merchant_state" als "Unknown" kennzeichnen
df["merchant_state"] = df["merchant_state"].fillna("Unknown")

### Vollständigkeit

In [10]:
gesamt_zellen = df.shape[0] * df.shape[1]
fehlende_zellen = df.isna().sum().sum()
vollstaendigkeit = (
    (gesamt_zellen - fehlende_zellen) / gesamt_zellen) * 100
print(f"Datenvollständigkeit: {vollstaendigkeit:.2f}%")

# Quelle: Ambach, D. data analytics master.
# Abgerufen am 10.03.2026 von
# https://github.com/dan-am/data analytics master/
# blob/main/3 data preparation/data validation.py

Datenvollständigkeit: 100.00%


### **Irrelevante Spalten löschen**

Eine Begründung, warum diese Spalten gelöscht werden, steht in der Datei **[01_Business_Data_Understanding](01_Business_Data_Understanding.ipynb)**.

In [11]:
df = df.drop(
    columns=[
        "merchant_id",
        "id_cards",
        "id_user",
        "card_number",
        "expires",
        "cvv",
        "card_on_dark_web",
        "mcc",
        "card_type",
        "has_chip",
        "acct_open_date",
        "address",
        "latitude",
        "longitude",
        "current_age",
        "retirement_age",
        "merchant_city",
        "use_chip",
    ]
)

### **Kategorische Variablen**

In [12]:
# Kategorische Variablen bestimmen
categorical_vars = [
    col for col in df
    if df[col].dtype == "object"
]

# Eindeutige Werte der kategorischen Variablen ausgeben
for i in categorical_vars:
    print(i)
    print(df[i].unique())
    print("")

date
['2010-01-01 00:01:00' '2010-01-01 00:02:00' '2010-01-01 00:05:00' ...
 '2019-10-31 23:57:00' '2019-10-31 23:58:00' '2019-10-31 23:59:00']

amount
['$-77.00' '$14.57' '$80.00' ... '$397.54' '$693.96' '$694.30']

merchant_state
['ND' 'IA' 'CA' 'IN' 'MD' 'NY' 'Unknown' 'TX' 'HI' 'PA' 'WI' 'GA' 'AL'
 'CT' 'WA' 'MA' 'CO' 'NJ' 'OK' 'MT' 'FL' 'AZ' 'KY' 'LA' 'IL' 'OH' 'MO'
 'MI' 'KS' 'NC' 'AR' 'TN' 'NM' 'SC' 'MN' 'NV' 'OR' 'VA' 'SD' 'WV' 'ME'
 'MS' 'RI' 'NH' 'DE' 'VT' 'Mexico' 'ID' 'NE' 'DC' 'UT' 'Vatican City' 'WY'
 'Dominican Republic' 'Canada' 'AK' 'Costa Rica' 'Germany' 'China'
 'United Kingdom' 'Estonia' 'Tuvalu' 'Taiwan' 'United Arab Emirates'
 'Lithuania' 'Netherlands' 'Japan' 'Greece' 'Vietnam' 'Haiti' 'Ireland'
 'Singapore' 'France' 'South Africa' 'Thailand' 'Italy' 'Denmark'
 'Jamaica' 'Benin' 'Belgium' 'Sierra Leone' 'Indonesia' 'Colombia'
 'Switzerland' 'Portugal' 'New Zealand' 'Jordan' 'Guatemala' 'Hong Kong'
 'Finland' 'Mongolia' 'Saudi Arabia' 'Philippines' 'Norway' 'Hunga

Die Variable **date** wird in das Datenformat Datum geändert und dabei wird ausschließlich der Tag ohne die Uhrzeitangabe (yyy-mm-dd) als separate Variable betrachtet.

Bei den Variablen **amount**, **credit_limit**,  **per_capita_income**,  **yearly_income** und **total_debt** wird das Dollarzeichen entfernt und als numerische Variable gespeichert.

Für die Variablen **description**, **errors**,  **merchant_state** werden Kategorien gebildet, um die Werte zu gruppieren und die Auswertung zu vereinfachen.


In [13]:
# Variable "date" als Datetime speichern
df["date"] = pd.to_datetime(df["date"])

# Tag aus dem Variable "date" extrahieren
df["day"] = df["date"].dt.date

In [14]:
# Dollarzeichen entfernen und als numerische Spalten speichern
list_dollar_change = [
    "amount",
    "credit_limit",
    "per_capita_income",
    "yearly_income",
    "total_debt",
]

for i in list_dollar_change:
    # Dollarzeichen entfernen
    df[i] = df[i].str[1:]
    # In numerische Spalte umwandeln
    df[i] = pd.to_numeric(df[i], errors="coerce")

In [15]:
# Kategorien für "description" erstellen
df["description_category"] = df["description"].apply(
    lambda x: get_key_value(x, MAP_DESCRIPTION)
)
df = df.drop(columns=["description"])
df["description_category"].value_counts()

description_category
Food_and_Beverage               19085160
Automotive                       9344898
Retail_General                   5460328
Travel_and_Transportation        3593642
Health_and_Medical               3402133
Financial_and_Legal              2404938
Electronics_and_Technology       1418134
Home_and_Garden                  1262886
Entertainment_and_Recreation     1073906
Books_and_Media                   933437
Utilities                         915159
Professional_Services             662748
Beauty_and_Personal_Care          559090
Metals_and_Manufacturing          486750
Clothing_and_Apparel              448529
Lodging                            63599
Name: count, dtype: int64

In [16]:
# Kategorien für "merchant_state" erstellen
df["merchant_category"] = df["merchant_state"].apply(
    lambda x: get_key_value(x, MAP_MERCHANT_STATE)
)
df = df.drop(columns=["merchant_state"])
df["merchant_category"].value_counts()

merchant_category
US_States        44833627
North_America      172617
Europe             111304
Asia                48696
South_America        9773
Africa               4099
Oceania              3317
Name: count, dtype: int64

In [17]:
# Kategorien für "errors" erstellen
df["error_category"] = df["errors"].apply(
    lambda x: get_key_value(x, MAP_ERROR_CATEGORY)
)
df = df.drop(columns=["errors"])
df["error_category"].value_counts()

error_category
No_Error           50306417
Single_Error         805395
Multiple_Errors        3525
Name: count, dtype: int64

### **Numerischen Variablen**

In [18]:
# Numerische Variablen ausgeben
numerical_vars = [
    col for col in df
    if df[col].dtype != "object"
]
for i in range(0, len(numerical_vars), 10):
    print(", ".join(numerical_vars[i:i + 10]))

id_transaction, date, client_id, card_id, amount, zip, num_cards_issued, credit_limit, year_pin_last_changed, birth_year
birth_month, per_capita_income, yearly_income, total_debt, credit_score, num_credit_cards


Die Variable **zip**, **birth_year**, **birth_month** werden als kategorische Variable gespeichert.

In [19]:
list_numeric_to_string = ["zip", "birth_year", "birth_month"]
for i in list_numeric_to_string:
    df[i] = df[i].astype(str)  # Umwandlung in String
    print(df[i].value_counts())  # Anzahl der Werte anzeigen

zip
Unknown    6281710
98516       164571
95687       155919
29229       150235
38606       130334
            ...   
80217            1
14233            1
77831            1
52651            1
87832            1
Name: count, Length: 25257, dtype: int64
birth_year
1970    1812556
1972    1716533
1967    1661377
1968    1537799
1977    1334256
         ...   
1992      98479
1994      88274
1918      61648
1995      50396
1996       4330
Name: count, Length: 74, dtype: int64
birth_month
11    5329686
1     4896101
2     4617258
3     4585172
10    4451062
9     4449814
12    4323215
5     4117146
7     4061922
8     3687125
4     3593453
6     3003383
Name: count, dtype: int64


Bestimmung der Outlier

In [20]:
# Neue Bestimmung der numerischen Werte
# (wg. vorheriger Umwandlung in kategorische Variablen)
numerical_vars = [
    col for col in df
    if df[col].dtype != "object"
]

# Outlier mit der IQR-Methode bestimmen
for i in numerical_vars:
    var_q1 = df[i].quantile(0.25)
    var_g3 = df[i].quantile(0.75)
    var_iqr = var_g3 - var_q1
    # Anzahl der Outlier
    n_outlier = (
        (df[i] < var_q1 - 1.5 * var_iqr)
        | (df[i] > var_g3 + 1.5 * var_iqr)).sum()
    # Prozentanteil der Outlier
    p_outlier = round(n_outlier / len(df) * 100, 1)
    print(
        f"Variable {i} hat {n_outlier} Ausreißer, "
        f"was {p_outlier} Prozent entspricht"
    )

Variable id_transaction hat 0 Ausreißer, was 0.0 Prozent entspricht
Variable date hat 0 Ausreißer, was 0.0 Prozent entspricht
Variable client_id hat 0 Ausreißer, was 0.0 Prozent entspricht
Variable card_id hat 0 Ausreißer, was 0.0 Prozent entspricht
Variable amount hat 4092799 Ausreißer, was 8.0 Prozent entspricht
Variable num_cards_issued hat 0 Ausreißer, was 0.0 Prozent entspricht
Variable credit_limit hat 1781394 Ausreißer, was 3.5 Prozent entspricht
Variable year_pin_last_changed hat 130211 Ausreißer, was 0.3 Prozent entspricht
Variable per_capita_income hat 3086297 Ausreißer, was 6.0 Prozent entspricht
Variable yearly_income hat 2641763 Ausreißer, was 5.2 Prozent entspricht
Variable total_debt hat 833089 Ausreißer, was 1.6 Prozent entspricht
Variable credit_score hat 1303847 Ausreißer, was 2.6 Prozent entspricht
Variable num_credit_cards hat 216225 Ausreißer, was 0.4 Prozent entspricht


Vor dem Hintergrund, dass es sich um ein Clustering-Projekt handelt, werden die Ausreißer nicht weiter behandelt. Die Ausreißer in den betroffenen Variablen stellen reales Verhalten der Kunden dar. Eine Entfernung dieser Datenpunkte würde die Clustering-Ergebnisse verfälschen.

### **Cleaned-Datensatz**

In [21]:
print("Cleaned-Datensatz:", df.shape)
display(df.head())
df.info()

Cleaned-Datensatz: (51115337, 22)


,id_transaction,date,client_id,card_id,amount,zip,card_brand,num_cards_issued,credit_limit,year_pin_last_changed,birth_year,birth_month,gender,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,day,description_category,merchant_category,error_category
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,58523,Mastercard,2,24772,2010,1989,7,Female,23679,48277,110153,740,4,2010-01-01,Food_and_Beverage,US_States,No_Error
1,7475327,2010-01-01 00:01:00,1556,2972,-77.00,58523,Visa,2,15300,2020,1989,7,Female,23679,48277,110153,740,4,2010-01-01,Food_and_Beverage,US_States,No_Error
2,7475327,2010-01-01 00:01:00,1556,2972,-77.00,58523,Mastercard,2,55,2008,1989,7,Female,23679,48277,110153,740,4,2010-01-01,Food_and_Beverage,US_States,No_Error
3,7475327,2010-01-01 00:01:00,1556,2972,-77.00,58523,Amex,2,11200,2020,1989,7,Female,23679,48277,110153,740,4,2010-01-01,Food_and_Beverage,US_States,No_Error
4,7475328,2010-01-01 00:02:00,561,4575,14.57,52722,Mastercard,1,26960,2010,1971,6,Male,18076,36853,112139,834,5,2010-01-01,Retail_General,US_States,No_Error


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51115337 entries, 0 to 51115336
Data columns (total 22 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   id_transaction         int64         
 1   date                   datetime64[ns]
 2   client_id              int64         
 3   card_id                int64         
 4   amount                 float64       
 5   zip                    object        
 6   card_brand             object        
 7   num_cards_issued       int64         
 8   credit_limit           int64         
 9   year_pin_last_changed  int64         
 10  birth_year             object        
 11  birth_month            object        
 12  gender                 object        
 13  per_capita_income      int64         
 14  yearly_income          int64         
 15  total_debt             int64         
 16  credit_score           int64         
 17  num_credit_cards       int64         
 18  day                 

***
## Phase 5: Feature Engineering
***
Aus dem aufbereiteten Datensatz werden zwei Teildatensätze erstellt: eine Zeitreihe mit den Variablen card_id, day und amount, die als Grundlage für das Clustering dient, sowie ein ergänzender Datensatz mit den übrigen Variablen. Der Datensatz mit den ergänzenden Informationen wird im Anschluss herangezogen, um die gebildeten Cluster inhaltlich zu analysieren und die Ergebnisse zu bewerten.


### **Zeitreihen**

Für alle Karten-ID wird für jeden Tag ein Eintrag erstellt (vollständige Zeitreihe)


In [22]:
# Datensatz: Pro Karten-ID Summe der Umsätze (amount) pro Tag (day)
df_timeseries_pre = (
    df.groupby(["card_id", "day"])
    .agg(amount_sum=("amount", "sum"))
    .reset_index()
)

# Tag in ein Datetime-Format bringen
df_timeseries_pre["day"] = pd.to_datetime(df_timeseries_pre["day"])

# Liste aller Karten-IDs
list_cards = df_timeseries_pre["card_id"].unique()
# Liste aller Tage zwischen den Min-Tag und Max-Tag die im Datensatz vorkommen
list_days = pd.date_range(
    df_timeseries_pre["day"].min(),
    df_timeseries_pre["day"].max(),
)

# Vollständige Karte-Tag-Kombinationen erstellen
combination_card_day = pd.MultiIndex.from_product(
    [list_cards, list_days],
    names=["card_id", "day"]
)

# Erstellen eines Datensatz, bei dem jede Karte für jeden Tag
# den tatsächlichen Umsatz oder den Wert 0 hat
df_timeseries = (
    df_timeseries_pre.set_index(["card_id", "day"])
    .reindex(combination_card_day, fill_value=0)
    .reset_index()
)

df_timeseries.head()


,card_id,day,amount_sum
0,0,2010-01-01,653.68
1,0,2010-01-02,64.80
2,0,2010-01-03,450.76
3,0,2010-01-04,0.00
4,0,2010-01-05,122.84


In [23]:
df_timeseries.shape

(14618961, 3)

Die Karten-IDs, bei denen mehr als 50 Prozent der Tageswerte den Betrag 0 aufweisen, werden aus dem Datensatz entfernt, da sie zu wenig Transaktionsaktivität zeigen und somit nur eine geringe Aussagekraft für das Clustering besitzen.

In [24]:
# # Karten-IDs mit mehr als 50 % Null-Werten ermitteln und herausfiltern
df_timeseries_50 = (
    df_timeseries
    .assign(zero=df_timeseries["amount_sum"].eq(0))  # True/False für 0-Wert
    .groupby("card_id")
    .agg(
        days_total=("day", "nunique"),   # Anzahl Tage
        days_zero=("zero", "sum"),        # True zählt als 1
    )
    .assign(
        percent_zero=lambda x: 100 * x["days_zero"] / x["days_total"]
    )
    .sort_values("percent_zero", ascending=False)
)

card_ids_under_50 = (
    df_timeseries_50
    .loc[df_timeseries_50["percent_zero"] <= 50]
    .index
    .tolist()
)

df_timeseries = df_timeseries[
    df_timeseries["card_id"].isin(card_ids_under_50)
]

In [25]:
# Liste der Karten-IDs
list_card_ids = df_timeseries["card_id"].unique()

In [26]:
# Pivotisierung: jede Zeile repräsentiert eine Karte
# Fehlende Werte werden mit 0 gefüllt
df_timeseries = df_timeseries.pivot(
    index="card_id", columns="day", values="amount_sum"
).fillna(0)

In [27]:
df_timeseries.shape

(1786, 3591)

In [28]:
df_timeseries.head(3)

day      2010-01-01  2010-01-02  2010-01-03  2010-01-04  2010-01-05  \
card_id                                                               
0            653.68        64.8      450.76        0.00      122.84   
1            533.68         0.0      545.44      409.68      164.24   
2           2814.46        81.8      125.46      989.74       30.96   

day      2010-01-06  2010-01-07  2010-01-08  2010-01-09  2010-01-10  \
card_id                                                               
0            289.32      183.60        0.00        0.00      513.08   
1            341.96        0.00        0.00      951.08      549.20   
2            162.14     1346.08       90.68     -539.42      369.76   

day      2010-01-11  2010-01-12  2010-01-13  2010-01-14  2010-01-15  \
card_id                                                               
0              0.00       35.28        0.00         0.0        0.00   
1            175.20        0.00        0.00         0.0      201.28   
2             78.52       70.32      357.38       118.2      811.50   

day      2010-01-16  2010-01-17  2010-01-18  2010-01-19  2010-01-20  \
card_id                                                               
0               0.0      494.88         0.0       88.56       68.88   
1               0.0      357.48       419.4     1346.64      273.20   
2            -375.1       74.10       255.4      591.68      145.70   

day      2010-01-21  2010-01-22  2010-01-23  2010-01-24  2010-01-25  \
card_id                                                               
0            722.68        0.00        0.00        0.00        0.00   
1            227.52      302.16      559.68      397.88      162.64   
2            407.26       22.50        0.00      105.38      255.00   

day      2010-01-26  2010-01-27  2010-01-28  2010-01-29  2010-01-30  \
card_id                                                               
0            467.16        0.00      461.20      247.36        0.00   
1            765.72      322.44       12.20      168.28      142.96   
2            268.76       29.02       82.74       71.62       28.74   

day      2010-01-31  2010-02-01  2010-02-02  2010-02-03  2010-02-04  \
card_id                                                               
0            103.40       53.12       80.08        0.00       46.80   
1            415.68      359.60      654.96      720.52      361.40   
2            544.48      334.60      215.66       71.98       70.14   

day      2010-02-05  2010-02-06  2010-02-07  2010-02-08  2010-02-09  \
card_id                                                               
0             51.32        0.00      449.76       64.04        0.00   
1            999.32      506.16      365.32        8.88      570.48   
2             92.26      479.98      350.44        1.72      384.32   

day      2010-02-10  2010-02-11  2010-02-12  2010-02-13  2010-02-14  \
card_id                                                               
0              0.00        0.00        0.00        0.00      444.60   
1            523.48      193.64      179.24      289.24      241.28   
2              2.72       30.74      306.20       97.82       80.16   

day      2010-02-15  2010-02-16  2010-02-17  2010-02-18  2010-02-19  \
card_id                                                               
0              0.00      429.52        0.00     1460.00        0.00   
1            163.52      100.04      161.84      175.40      368.56   
2            154.54       40.12      681.66      317.72      390.06   

day      2010-02-20  2010-02-21  2010-02-22  2010-02-23  2010-02-24  \
card_id                                                               
0           -656.80       63.52        0.00        0.00        0.00   
1            188.64        0.00      480.00        0.00     1169.88   
2            223.20       64.00      119.52       70.92        0.00   

day      2010-02-25  2010-02-26  2010-02-27  2010-02-28  2010-03-01  \
card_id      

In [29]:
# Daten speichern
path_temp = INPUT_PATH / "data_timeseries.csv"
df_timeseries.to_csv(path_temp, index=True)

### **Zusatzinfos**
Ergänzend wird ein Zusatzinfo-Datensatz erstellt, der die übrigen Variablen pro Karte zusammenfasst. Dabei werden numerische Spalten durch ihren Median und kategoriale Spalten durch den jeweils häufigsten Wert aggregiert. Die Ergebnisse werden anschließend zu einem Datensatz zusammengeführt, sodass zu jede card_id eindeutig zusammengefassten Informationen vorliegen.

In [30]:
df_add_info = df.drop(["date", "id_transaction", "amount", "day"], axis=1)

# Nur Karten-IDs, die in der Zeitreihe sind
df_add_info = df_add_info[df_add_info["card_id"].isin(list_card_ids)]

In [31]:
# Numerische Spalten aggregieren (Median)
numerical_vars = df_add_info.select_dtypes(include="number").columns
df_num = df_add_info.groupby("card_id")[numerical_vars].median()

# Kategorische Spalten aggregieren (häufigster Wert)
categorical_vars = df_add_info.select_dtypes(exclude="number").columns
df_cat = df_add_info.groupby("card_id")[categorical_vars].agg(
    lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan
)

# Numerische und kategorische Ergebnisse zusammenführen
df_zusatzinfo = df_num.join(df_cat)

# card_id Spalte löschen, da diese Spalte als Index gespeichert ist
df_zusatzinfo = df_zusatzinfo.drop("card_id", axis=1)

In [32]:
df_zusatzinfo.head(3)

,client_id,num_cards_issued,credit_limit,year_pin_last_changed,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,zip,card_brand,birth_year,birth_month,gender,description_category,merchant_category,error_category
card_id,,,,,,,,,,,,,,,,,
0,1362.0,2.0,30450.0,2014.0,35563.0,72510.0,44317.0,727.0,4.0,22015,Mastercard,1962,1,Male,Food_and_Beverage,US_States,No_Error
1,550.0,2.0,7900.0,2012.0,21219.0,30248.0,35766.0,763.0,4.0,98366,Mastercard,1944,2,Male,Retail_General,US_States,No_Error
2,556.0,1.0,21818.0,2011.0,17856.0,36405.0,31815.0,715.0,2.0,44203,Mastercard,1973,12,Male,Food_and_Beverage,US_States,No_Error


**Hinweis**: Die Skalierung der Daten erfolgt im Skript **[03_Modeling_Evaluation](03_Modeling_Evaluation.ipynb)**., da dort jeweils 100 zufällig gewählte Zeitreihen in fünf Läufen gewählt werden. Die Skalierung erfolgt dann auf die ausgewählten Zeitreihen.

In [33]:
# Daten speichern
path_temp = INPUT_PATH / "data_add_info.csv"
df_zusatzinfo.to_csv(path_temp, index=True)

***
### **Profilingreport**

In [34]:
# Profilingreport der Zusatzinfos laden
pr = ProfileReport(df_zusatzinfo, title="Financial Transactions Data")
filename_pr = "../output/add_info_data_pr.html"
path_pr = os.path.abspath(filename_pr)

pr.to_file(path_pr)  # ProfileReport als HTML speichern
webbrowser.open(f"file://{path_pr}")  # ProfileReport im Browser öffnen

python(89068) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(89069) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 17/17 [00:00<00:00, 189.51it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

python(89079) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


True

***
***